# Import data into QuickBooks Online

Load the existing CloudFlow CSVs into the configured QuickBooks sandbox in this order: **Chart of Accounts, Customers, then Journal Entries**.

Run cells from top to bottom. `APPLY_IMPORT = False` validates locally without API calls. The configuration below currently has `APPLY_IMPORT = True`, which enables API writes when the execution cell runs. Set it to `True` and rerun the configuration and execution cells when ready to create records. Authentication uses the existing `.env` and `tokens/qbo_tokens.json`; Azure SQL is not needed.

Reusable logic lives in `src/qbo_import.py`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "qbo_import.py").exists():
    raise FileNotFoundError("Open this notebook from the project root or notebooks folder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.qbo_import import load_import_data, QBOClient, import_records

print("Project root:", PROJECT_ROOT)

## Configure the import

In [ ]:
DATA_DIR = PROJECT_ROOT / "Data"
ACCOUNT_FILE = "qbo_chart_of_accounts.csv"
CUSTOMER_FILE = "customer_master.csv"
JOURNAL_FILES = [
    "qbo_journal_import_part1.csv",
    "qbo_journal_import_part2.csv",
]
APPLY_IMPORT = True

# These four files are loaded. Set CUSTOMER_FILE = None to skip customers.

## Validate and preview

Validation checks required columns, unique identities, account/customer references, amounts, and balanced journals. Customer_ID maps to QBO DisplayName; populated source attributes are preserved in Notes. QBO assigns its own internal customer Id. MRR values remain descriptive notes and do not create transactions. Churn_Date does not deactivate a customer. A nonempty journal Name must match a Customer_ID.

CSV detail labels map to US API values. The two Other Expense accounts use OtherMiscellaneousExpense to preserve their classification. Nonempty TaxCode, Location, and Class are not supported yet and cause validation to fail.

In [ ]:
accounts, customers, journals = load_import_data(
    DATA_DIR, account_file=ACCOUNT_FILE, journal_files=JOURNAL_FILES, customer_file=CUSTOMER_FILE
)

summary = pd.DataFrame([
    {"Entity": "Account", "Records": len(accounts)},
    {"Entity": "Customer", "Records": len(customers)},
    {"Entity": "JournalEntry", "Records": len(journals)},
    {"Entity": "Journal lines", "Records": sum(len(j["Line"]) for j in journals)},
])
display(summary)
print("All source journals are balanced. No API calls made.")

In [ ]:
display(pd.DataFrame(accounts).head())
display(pd.DataFrame(customers).head())
display(pd.DataFrame([
    {"Journal": j["DocNumber"], "Date": j["TxnDate"], "Lines": len(j["Line"])}
    for j in journals
]))

## Execute

This cell reloads and validates the files, then checks existing QuickBooks records before creating anything. Matching records are reused; conflicts or inactive records stop the import. No existing records are updated or deleted.

Imports are not atomic: successful records remain after a later failure. Rerun unchanged inputs to resume. Avoid concurrent imports. Stable request IDs protect identical retries; after a sandbox reset, old request IDs may still be replayed. API acceptance of locale-specific account types is verified only during import.

In [ ]:
import_results = []
if not APPLY_IMPORT:
    print("Validation-only mode. Set APPLY_IMPORT = True above to import into QuickBooks.")
else:
    accounts, customers, journals = load_import_data(
        DATA_DIR, account_file=ACCOUNT_FILE, journal_files=JOURNAL_FILES, customer_file=CUSTOMER_FILE
    )
    client = QBOClient()
    try:
        import_results = import_records(accounts, customers, journals, client)
    finally:
        client.session.close()
    print("Import complete.")

In [ ]:
if import_results:
    result_df = pd.DataFrame(import_results)
    display(result_df.groupby(["entity", "status"]).size().rename("records").reset_index())
else:
    print("No import results: validation-only mode or execution has not completed.")

After a successful import, run `01_extract_quickbooks_bronze.ipynb` or the pipeline notebook to refresh the analytical data.

Reference: [Intuit API best practices and request IDs](https://blogs.a.intuit.com/2018/09/10/quickbooks-online-api-best-practices/).